# AnyVisLoc: аудит даних і причин відмов покриття

**Результат:** перелік перевірених сцен і кадрів, пояснення відмов, незалежний резерв та окремий контрольний дослід.

Використовуємо SIFT + аерофотокарту. Планувальник і навчання моделей не змінюються. Реальні дані обробляємо лише в **hosted Google Colab**.

Початковий запуск Scene_05: 20 кадрів, 4 прийняті прив’язки, 16 відмов до SIFT через покриття. Його файли зберігаються без змін.

Перед запуском **вручну скопіюйте** актуальні `scripts/`, `configs/`, `requirements.txt` і цей notebook до каталогу проєкту у Drive. Автоматичного завантаження коду немає. Докладний посібник: `docs/vml_data_audit.md`.


In [1]:
import os
from pathlib import Path
if not os.environ.get('COLAB_RELEASE_TAG') or not Path('/content').is_dir():
    raise RuntimeError('Use a hosted Google Colab runtime, not a local runtime.')
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Шляхи, залежності та початковий запуск

`BASELINE_ROOT` має містити оригінальні `run.json`, `manifest.json` і всю папку `shards` із запуску v3. Тексту summary недостатньо для перевірки prior і контрольних сум.

Результати аудиту та контролю мають **різні нові каталоги**. Бюджет архівів у Drive — 5 GB = 5 000 000 000 байтів; результати також потребують місця. Розпаковані дані зберігаються на диску Colab, не у Drive та не в RAM.


In [2]:
REPO_ROOT = Path('/content/drive/MyDrive/PhD/proj')
ARCHIVE_ROOT = REPO_ROOT / 'AnyVisLoc'
BASELINE_ROOT = REPO_ROOT / 'vml/anyvisloc-smoke-v3'
AUDIT_ROOT = REPO_ROOT / 'vml/anyvisloc-audit-v1'
CONTROL_ROOT = REPO_ROOT / 'vml/anyvisloc-coverage-control-v1'
CONFIG_PATH = REPO_ROOT / 'configs/chance_constrained/vml/anyvisloc_smoke.v1.json'
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(REPO_ROOT / 'requirements.txt')])


0

In [3]:
import json, importlib
sys.path.insert(0, str(REPO_ROOT))
importlib.invalidate_caches()
prefix = 'scripts.pipeline.chance_constrained.visual_map_localization'
for name in ('anyvisloc', 'planar', 'smoke_localizer', 'anyvisloc_smoke', 'audit'):
    importlib.reload(importlib.import_module(prefix + '.' + name))
audit = importlib.import_module(prefix + '.audit')
config = audit.read_json(CONFIG_PATH)
baseline = audit.load_baseline(BASELINE_ROOT, config)
print('Baseline run:', baseline['run_identity'])
print('Retained aerial frames:', len(baseline['trials']))
print('Baseline versions:', {k: audit.read_json(BASELINE_ROOT / 'run.json')[k] for k in ('numpy', 'opencv')})


Baseline run: cf5ffd0ebdaebbc7eb9434b4caa2cf1c97bb9f3cab09ee5dd3098f125771b5d0
Retained aerial frames: 20
Baseline versions: {'numpy': '2.1.3', 'opencv': '4.14.0'}


## 2. Перелік архівів і бюджет

Читання ZIP-каталогу та SHA-256 ще **не означає**, що сцена перевірена. Повна перевірка — після розпакування та читання всіх кадрів. Пошкоджений ZIP позначається помилкою.

Якщо бюджет перевищений, самостійно впорядкуйте зберігання та повторіть клітинку. Нічого не видаляється автоматично. Перед звільненням місця збережіть можливість повторно отримати ті самі дані: контрольні суми не замінюють архіви.


In [5]:
inventory = audit.inventory_archives(ARCHIVE_ROOT, AUDIT_ROOT / 'archive_inventory')
for item in inventory['archives']:
    print(item)
print('Archive storage, GB:', inventory['archive_bytes'] / 1e9)
print('Free storage, GB:', inventory['free_bytes'] / 1e9)
if not inventory['within_archive_budget']:
    raise RuntimeError('Archives exceed the 5 GB budget. No files are deleted automatically.')


{'archive': '/content/drive/MyDrive/PhD/proj/AnyVisLoc/Scene_05.zip', 'archive_bytes': 2347972708, 'source_available': True, 'fully_verified': False, 'sha256': '0ad83a7fa97fd9f7e14dbbfacdb1146d548fd3aa82b92a76a19463a7dca631b4', 'unpacked_bytes': 2354754179, 'scene_ids': ['Scene_05'], 'status': 'discovered_not_audited'}
Archive storage, GB: 2.347972708
Free storage, GB: 3.709227008


## 3. Одна сцена за раз: розпакування або повторне використання

Вкажіть архів і залиште той самий `STAGING_TAG`, якщо сцена вже розпакована. Код перевірить SHA-256 архіву, а також розміри й CRC наявних файлів. CRC — контрольна сума, записана в ZIP для перевірки цілісності файла. Якщо все збігається, використає їх повторно (`status: reused`). Якщо бракує файлів, розпакує лише відсутні (`resumed`). Завершені файли не перезаписуються. При відмінності наявного файлу зупиниться з поясненням; тоді можна вибрати новий тег, зберігши попередню папку.

Перевірка великого архіву в Drive може тривати кілька хвилин. Друга копія ZIP не створюється, місце перевіряється лише для відсутніх файлів із запасом 512 MiB. Архіви та попередні результати автоматично не видаляються.

Для нового аудиту починайте зі Scene_05, потім повторюйте розділи 3–4 для інших сцен. Якщо розподіл уже зафіксований у розділі 5, повторний запуск розділу 4 пропустить аудит. Розділ 5 відновить звіт зі збережених результатів без нового розподілу сцен.


In [6]:
ARCHIVE_NAME = 'Scene_05.zip'  # Change for the next scene.
STAGING_TAG = 'v1'  # Keep this tag to verify and reuse existing staging.
source = next((r for r in inventory['archives'] if Path(r['archive']).name == ARCHIVE_NAME), None)
if source is None or source['status'] != 'discovered_not_audited':
    raise ValueError('A successful inventory entry is required for this archive.')
DATASET_ROOT = Path('/content/anyvisloc-audit') / (Path(ARCHIVE_NAME).stem + '-' + STAGING_TAG)
print('Verifying archive and existing staging; large files may take several minutes.')
receipt = audit.stage_archive(source['archive'], DATASET_ROOT,
                             AUDIT_ROOT / 'staging_receipts', source['sha256'])
print(receipt)


Verifying archive and existing staging; large files may take several minutes.
{'archive': '/content/drive/MyDrive/PhD/proj/AnyVisLoc/Scene_05.zip', 'sha256': '0ad83a7fa97fd9f7e14dbbfacdb1146d548fd3aa82b92a76a19463a7dca631b4', 'archive_bytes': 2347972708, 'unpacked_bytes': 2354754179, 'destination': '/content/anyvisloc-audit/Scene_05-v1', 'verified_existing_files': 1123, 'extracted_files': 0, 'status': 'reused'}


## 4. Повний аудит без запуску SIFT

Перевіряються всі кадри. Кадри поза AGL 20–120 м зберігаються у таблиці з причиною виключення. Помилка одного кадру не зупиняє наступні. Спільна помилка карти або reference записується на рівні сцени.

**Prior** — оцінка XY до зіставлення. Тут це еталон + штучна гаусова похибка зі стандартним відхиленням 20 м по осі. Це не запис VIO. **99%** задає область невизначеності (приблизно 61 м радіуса), а **95%** — окремий поріг площі пошукового прямокутника, покритої картою.

Три показники:

1. `footprint_coverage`: частка фактично видимої ділянки на карті, перетин полігонів.
2. `gt_search_coverage`: покриття пошукового прямокутника навколо еталона з чинною невизначеністю.
3. `prior_search_coverage`: покриття пошукового прямокутника навколо фактичного prior.

Для початкових кадрів prior береться зі збережених результатів. Перевіряються конфігурація, SHA-256 джерел, AGL і збіг покриття. Невідповідність блокує контрольний дослід до з’ясування причин.

`usable_frames.csv` містить геометрично підтримувані кадри незалежно від порога 95% та успіху SIFT. `frames.csv` містить усі кадри. `map-inspection.png` показує підозрілі чорні/білі ділянки, але **не визначає nodata**. Проходження геометрії не доводить відсутності викривлень від рельєфу чи забудови.


In [ ]:
if (AUDIT_ROOT / 'manifest.json').exists():
    print('Audit is already frozen. Existing scene results are preserved.')
    print('Skip re-auditing and continue to section 5 to rebuild the report.')
else:
    scenes = audit.audit_dataset(DATASET_ROOT, AUDIT_ROOT, config, BASELINE_ROOT)
    for scene in scenes:
        print({key: scene[key] for key in ('scene_id', 'status', 'fully_verified', 'frame_count',
                                         'data_valid_count', 'geometry_supported_count',
                                         'strict_policy_pass_count', 'shared_errors', 'factor_counts')})
    print('Results:', AUDIT_ROOT / 'scenes')


## 5. Фіксація сцен і відновлення звіту

Для нового експерименту запускайте після аудиту всіх обраних сцен. Scene_05 — робоча; серед нових сцен із перевіреним комплектом `ceil(N/3)` призначаються для остаточної перевірки за порядком SHA-256 рядка `42:<scene_id>`. Результати зіставлення на вибір не впливають.

Якщо `manifest.json` уже є, розподіл не змінюється. Код перевірить контрольні суми збережених таблиць і знову створить англомовні `scenes.csv` та `report.md`. Це виправляє випадок, коли розподіл збережений, але `report.md` відсутній. Сцени й початковий запуск не перераховуються. Змінені таблиці викличуть помилку, а не приховану зміну експерименту.


In [7]:
manifest = audit.finalize_audit(AUDIT_ROOT)
print(json.dumps(manifest['roles'], ensure_ascii=False, indent=2))
print('Independent holdout formed:', manifest['reserve_formed'])
from IPython.display import display, Markdown
display(Markdown((AUDIT_ROOT / 'report.md').read_text(encoding='utf-8')))


{
  "Scene_05": "working",
  "Scene_07": "working",
  "Scene_21": "holdout"
}
Independent holdout formed: True


# AnyVisLoc audit: SIFT and aerial reference maps

The prior is a synthetic XY position estimate, not recorded VIO output.
95% is an engineering threshold on search rectangle area, not localization success probability.
Footprint coverage uses polygon intersection; search rectangles retain the original pixel rounding.
Near-black/white regions are inspection indicators, not established missing data.

## Scenes

- Scene_05: partially_usable; role working; frames 1118, geometry supported 787, strict policy passed 147. Shared errors: []; factors: {'agl_outside_20_120_m': 248, 'dsm_outside': 78, 'footprint_outside_map': 559, 'insufficient_map_margin_at_gt': 636, 'prior_shift_reduces_coverage': 347, 'rectified_size_unsupported': 5}.
- Scene_07: partially_usable; role working; frames 1219, geometry supported 209, strict policy passed 205. Shared errors: []; factors: {'agl_outside_20_120_m': 1006, 'dsm_outside': 3, 'footprint_outside_map': 4, 'insufficient_map_margin_at_gt': 6, 'prior_shift_reduces_coverage': 2, 'projection_crosses_horizon': 1}.
- Scene_21: partially_usable; role holdout; frames 1161, geometry supported 1082, strict policy passed 978. Shared errors: []; factors: {'agl_outside_20_120_m': 68, 'dsm_outside': 2, 'footprint_outside_map': 51, 'insufficient_map_margin_at_gt': 109, 'prior_shift_reduces_coverage': 64, 'projection_crosses_horizon': 5, 'rectified_size_unsupported': 4}.

## Baseline frames: explanations or unresolved evidence

- Scene_05/L05_0001.npz: accepted_xy_uncalibrated; coverage {'footprint_coverage': 1.0000000000000002, 'gt_search_coverage': 0.926233150644349, 'prior_search_coverage': 0.9539395734597157}; factors ['insufficient_map_margin_at_gt']; error None; baseline comparability verified: True.
- Scene_05/L05_0002.npz: insufficient_map_coverage; coverage {'footprint_coverage': 0.9999999999999999, 'gt_search_coverage': 0.886122569239835, 'prior_search_coverage': 0.9491750147318798}; factors ['insufficient_map_margin_at_gt']; error None; baseline comparability verified: True.
- Scene_05/L05_0003.npz: insufficient_map_coverage; coverage {'footprint_coverage': 1.0, 'gt_search_coverage': 0.8175085172567027, 'prior_search_coverage': 0.723300251814546}; factors ['insufficient_map_margin_at_gt', 'prior_shift_reduces_coverage']; error None; baseline comparability verified: True.
- Scene_05/L05_0004.npz: insufficient_map_coverage; coverage {'footprint_coverage': 0.9769224215612684, 'gt_search_coverage': 0.7458481613285883, 'prior_search_coverage': 0.7648279952550415}; factors ['footprint_outside_map', 'insufficient_map_margin_at_gt']; error None; baseline comparability verified: True.
- Scene_05/L05_0005.npz: insufficient_map_coverage; coverage {'footprint_coverage': 0.9323757069550103, 'gt_search_coverage': 0.6902191171800571, 'prior_search_coverage': 0.6341695776436964}; factors ['footprint_outside_map', 'insufficient_map_margin_at_gt', 'prior_shift_reduces_coverage']; error None; baseline comparability verified: True.
- Scene_05/L05_0006.npz: insufficient_map_coverage; coverage {'footprint_coverage': 0.826075989280576, 'gt_search_coverage': 0.5619986746189529, 'prior_search_coverage': 0.6035632223751508}; factors ['footprint_outside_map', 'insufficient_map_margin_at_gt']; error None; baseline comparability verified: True.
- Scene_05/L05_0007.npz: insufficient_map_coverage; coverage {'footprint_coverage': 0.9329449119012222, 'gt_search_coverage': 0.6235836907394308, 'prior_search_coverage': 0.6384160575235563}; factors ['footprint_outside_map', 'insufficient_map_margin_at_gt']; error None; baseline comparability verified: True.
- Scene_05/L05_0008.npz: insufficient_map_coverage; coverage {'footprint_coverage': 0.9318897133139924, 'gt_search_coverage': 0.6743568009747902, 'prior_search_coverage': 0.6641467651384617}; factors ['footprint_outside_map', 'insufficient_map_margin_at_gt', 'prior_shift_reduces_coverage']; error None; baseline comparability verified: True.
- Scene_05/L05_0009.npz: insufficient_map_coverage; coverage {'footprint_coverage': 0.9819911108205231, 'gt_search_coverage': 0.7741263225392754, 'prior_search_coverage': 0.714972747675537}; factors ['footprint_outside_map', 'insufficient_map_margin_at_gt', 'prior_shift_reduces_coverage']; error None; baseline comparability verified: True.
- Scene_05/L05_0010.npz: insufficient_map_coverage; coverage {'footprint_coverage': 0.9999923203146052, 'gt_search_coverage': 0.7887323943661971, 'prior_search_coverage': 0.7491300745650373}; factors ['footprint_outside_map', 'insufficient_map_margin_at_gt', 'prior_shift_reduces_coverage']; error None; baseline comparability verified: True.
- Scene_05/L05_0011.npz: insufficient_map_coverage; coverage {'footprint_coverage': 0.9763269313178631, 'gt_search_coverage': 0.7782891604507222, 'prior_search_coverage': 0.7827328995397556}; factors ['footprint_outside_map', 'insufficient_map_margin_at_gt']; error None; baseline comparability verified: True.
- Scene_05/L05_0012.npz: insufficient_map_coverage; coverage {'footprint_coverage': 1.0, 'gt_search_coverage': 0.8018363939899833, 'prior_search_coverage': 0.7667779632721202}; factors ['insufficient_map_margin_at_gt', 'prior_shift_reduces_coverage']; error None; baseline comparability verified: True.
- Scene_05/L05_0013.npz: insufficient_map_coverage; coverage {'footprint_coverage': 1.0, 'gt_search_coverage': 0.7955132343447385, 'prior_search_coverage': 0.8000322788896062}; factors ['insufficient_map_margin_at_gt']; error None; baseline comparability verified: True.
- Scene_05/L05_0014.npz: insufficient_map_coverage; coverage {'footprint_coverage': 1.0, 'gt_search_coverage': 0.7977239942298445, 'prior_search_coverage': 0.8052572527648661}; factors ['insufficient_map_margin_at_gt']; error None; baseline comparability verified: True.
- Scene_05/L05_0015.npz: insufficient_map_coverage; coverage {'footprint_coverage': 0.9999999999999998, 'gt_search_coverage': 0.8469652710159039, 'prior_search_coverage': 0.7693930542031808}; factors ['insufficient_map_margin_at_gt', 'prior_shift_reduces_coverage']; error None; baseline comparability verified: True.
- Scene_05/L05_0016.npz: insufficient_map_coverage; coverage {'footprint_coverage': 1.0000000000000002, 'gt_search_coverage': 0.9186526850969177, 'prior_search_coverage': 0.9407372100413092}; factors ['insufficient_map_margin_at_gt']; error None; baseline comparability verified: True.
- Scene_05/L05_0017.npz: accepted_xy_uncalibrated; coverage {'footprint_coverage': 1.0, 'gt_search_coverage': 1.0, 'prior_search_coverage': 1.0}; factors []; error None; baseline comparability verified: True.
- Scene_05/L05_0018.npz: accepted_xy_uncalibrated; coverage {'footprint_coverage': 1.0, 'gt_search_coverage': 1.0, 'prior_search_coverage': 0.9996683800364782}; factors ['prior_shift_reduces_coverage']; error None; baseline comparability verified: True.
- Scene_05/L05_0019.npz: insufficient_map_coverage; coverage {'footprint_coverage': 0.9999999999999998, 'gt_search_coverage': 1.0, 'prior_search_coverage': 0.9315687540348612}; factors ['prior_shift_reduces_coverage']; error None; baseline comparability verified: True.
- Scene_05/L05_0020.npz: accepted_xy_uncalibrated; coverage {'footprint_coverage': 0.9999999999999999, 'gt_search_coverage': 0.9987152722017023, 'prior_search_coverage': 1.0}; factors []; error None; baseline comparability verified: True.

## Interpretation

Footprint outside the map means part of the actual visible ground is unavailable. Insufficient margin at GT means the map cannot support the configured search uncertainty. Reduced coverage at the prior shows the contribution of its particular displacement. Factors may coexist.
Read/coordinate errors retain their exception details and require inspection; they do not establish poor texture as the cause.
Eligibility refers to the current protocol; it does not guarantee SIFT success or validity of the planar approximation.
Per-frame details: scenes/<scene>/frames.csv; geometry figures: geometry-*.png.
The control experiment is stored separately and does not change the production localization policy.


## 6. Окремий контроль: лише 16 відмов Scene_05

Єдина зміна: покриття нижче 95% не зупиняє спробу перед SIFT. Геометричні та ресурсні перевірки діють, пороги SIFT не змінюються. Це **не затверджує нову робочу політику**.

Scene_05 має бути розпакована зараз. За потреби повторіть її staging, вкажіть її каталог у `CONTROL_DATASET_ROOT`. Це окремий шлях, який не залежить від останньої перевіреної сцени. Повторювати аудит після фіксації не потрібно.

Потрібні початкові версії NumPy/OpenCV із baseline/run.json. Якщо версії відрізняються, відновіть їх у Colab, перезапустіть runtime й імпорти. Перевірка версій блокує непорівнянний запуск.

Перед зіставленням повторно перевіряються джерела всіх 20 початкових кадрів. Контроль виконує лише 16 coverage rejects. Окремі checksummed checkpoints дозволяють продовжити після розриву.

Прийняття SIFT не гарантує правильної прив’язки: дивіться XY-похибку відносно еталона. Частоти результатів цих 16 відібраних кадрів не є польотними ймовірностями.


In [8]:
RUN_CONTROL = True  # Enable after completing the audit.
# This directory must contain Scene_05/, regardless of the last scene staged.
CONTROL_DATASET_ROOT = Path('/content/anyvisloc-audit/Scene_05-v1')
if RUN_CONTROL:
    if not (CONTROL_DATASET_ROOT / 'Scene_05').is_dir():
        raise FileNotFoundError('Stage Scene_05 in section 3 and set CONTROL_DATASET_ROOT to its extraction directory.')
    results = audit.run_control(CONTROL_DATASET_ROOT, BASELINE_ROOT, AUDIT_ROOT, CONTROL_ROOT, config)
    print(audit.read_json(CONTROL_ROOT / 'summary.json'))
    display(Markdown((CONTROL_ROOT / 'report.md').read_text(encoding='utf-8')))
else:
    print('The control experiment has not been run.')


{'accepted_errors_m': [0.20163874560851694, 0.18822792939890118, 0.17620596005203806, 0.05756365025249149, 0.11078377990190966, 0.09677147812826768, 0.09673192449213187, 0.04009064909906857, 0.43636283992874, 0.2220536747951386, 0.0730256122053367, 0.6443577393255928, 0.21921439141141663, 0.1117141735626447, 0.40083092660120606, 0.05935006769810106], 'flight_probability_estimate': False, 'production_policy_changed': False, 'reasons': {'accepted_xy_uncalibrated': 16}, 'trial_count': 16}


# Control experiment: clipped search region

Only the response to coverage below 95% changed; this does not approve a new production policy.
SIFT acceptance does not establish a correct fix: inspect XY error against ground truth.

- Scene_05/L05_0002.npz: insufficient_map_coverage -> accepted_xy_uncalibrated; error, m: 0.20163874560851694; factors: ['insufficient_map_margin_at_gt'].
- Scene_05/L05_0003.npz: insufficient_map_coverage -> accepted_xy_uncalibrated; error, m: 0.18822792939890118; factors: ['insufficient_map_margin_at_gt', 'prior_shift_reduces_coverage'].
- Scene_05/L05_0004.npz: insufficient_map_coverage -> accepted_xy_uncalibrated; error, m: 0.17620596005203806; factors: ['footprint_outside_map', 'insufficient_map_margin_at_gt'].
- Scene_05/L05_0005.npz: insufficient_map_coverage -> accepted_xy_uncalibrated; error, m: 0.05756365025249149; factors: ['footprint_outside_map', 'insufficient_map_margin_at_gt', 'prior_shift_reduces_coverage'].
- Scene_05/L05_0006.npz: insufficient_map_coverage -> accepted_xy_uncalibrated; error, m: 0.11078377990190966; factors: ['footprint_outside_map', 'insufficient_map_margin_at_gt'].
- Scene_05/L05_0007.npz: insufficient_map_coverage -> accepted_xy_uncalibrated; error, m: 0.09677147812826768; factors: ['footprint_outside_map', 'insufficient_map_margin_at_gt'].
- Scene_05/L05_0008.npz: insufficient_map_coverage -> accepted_xy_uncalibrated; error, m: 0.09673192449213187; factors: ['footprint_outside_map', 'insufficient_map_margin_at_gt', 'prior_shift_reduces_coverage'].
- Scene_05/L05_0009.npz: insufficient_map_coverage -> accepted_xy_uncalibrated; error, m: 0.04009064909906857; factors: ['footprint_outside_map', 'insufficient_map_margin_at_gt', 'prior_shift_reduces_coverage'].
- Scene_05/L05_0010.npz: insufficient_map_coverage -> accepted_xy_uncalibrated; error, m: 0.43636283992874; factors: ['footprint_outside_map', 'insufficient_map_margin_at_gt', 'prior_shift_reduces_coverage'].
- Scene_05/L05_0011.npz: insufficient_map_coverage -> accepted_xy_uncalibrated; error, m: 0.2220536747951386; factors: ['footprint_outside_map', 'insufficient_map_margin_at_gt'].
- Scene_05/L05_0012.npz: insufficient_map_coverage -> accepted_xy_uncalibrated; error, m: 0.0730256122053367; factors: ['insufficient_map_margin_at_gt', 'prior_shift_reduces_coverage'].
- Scene_05/L05_0013.npz: insufficient_map_coverage -> accepted_xy_uncalibrated; error, m: 0.6443577393255928; factors: ['insufficient_map_margin_at_gt'].
- Scene_05/L05_0014.npz: insufficient_map_coverage -> accepted_xy_uncalibrated; error, m: 0.21921439141141663; factors: ['insufficient_map_margin_at_gt'].
- Scene_05/L05_0015.npz: insufficient_map_coverage -> accepted_xy_uncalibrated; error, m: 0.1117141735626447; factors: ['insufficient_map_margin_at_gt', 'prior_shift_reduces_coverage'].
- Scene_05/L05_0016.npz: insufficient_map_coverage -> accepted_xy_uncalibrated; error, m: 0.40083092660120606; factors: ['insufficient_map_margin_at_gt'].
- Scene_05/L05_0019.npz: insufficient_map_coverage -> accepted_xy_uncalibrated; error, m: 0.05935006769810106; factors: ['prior_shift_reduces_coverage'].


## Результати для спільного розбору

Збережіть виконаний notebook із виводами. Потрібні:

- з аудиту — `manifest.json`, `scenes.csv`, `report.md`;
- для кожної сцени — `scene.json`, `frames.csv/json`, `usable_frames.csv`, `geometry-*.png`, `map-inspection.png`;
- з контрольного досліду — `comparison.csv`, `summary.json`, `report.md`.

«Причина невідома» допустимо, якщо вказано, яких доказів бракує. Відмову покриття не називаємо поганою текстурою. Навіть відмова після запуску SIFT може пояснюватися розмиттям, відмінностями карти й кадру або геометричним наближенням.
